In [1]:
# Cell 1: Import libraries and load data
import pandas as pd
import numpy as np
from collections import defaultdict
import random
from tqdm import tqdm

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("="*60)
print("📦 STEP 2: NEGATIVE SAMPLING & DATA SPLITS")
print("="*60)

# Load the datasets
df_drugs = pd.read_csv('../data/processed/drugbank_extracted_cleaned.csv')
df_interactions = pd.read_csv('../data/processed/drugbank_interactions_typed.csv')

print(f"\n✅ Loaded drug data: {len(df_drugs)} drugs")
print(f"✅ Loaded interactions: {len(df_interactions):,} pairs")

print("\n" + "="*60)

📦 STEP 2: NEGATIVE SAMPLING & DATA SPLITS

✅ Loaded drug data: 17430 drugs
✅ Loaded interactions: 2,855,848 pairs



In [2]:
# Cell 2: Filter to drugs with both SMILES and interactions
print("="*60)
print("🔍 FILTERING TO USABLE DRUGS")
print("="*60)

# Get drugs that have SMILES
drugs_with_smiles = set(df_drugs[df_drugs['smiles'].notna()]['drugbank_id'])
print(f"\n✅ Drugs with SMILES: {len(drugs_with_smiles)}")

# Get drugs that appear in interactions
drugs_in_interactions = set(df_interactions['drug1'].unique()) | set(df_interactions['drug2'].unique())
print(f"✅ Drugs in interaction network: {len(drugs_in_interactions)}")

# Get intersection (drugs with BOTH)
usable_drugs = drugs_with_smiles & drugs_in_interactions
print(f"✅ Usable drugs (BOTH SMILES + interactions): {len(usable_drugs)}")

# Filter interactions to only include usable drugs
df_interactions_filtered = df_interactions[
    (df_interactions['drug1'].isin(usable_drugs)) & 
    (df_interactions['drug2'].isin(usable_drugs))
].copy()

print(f"\n🔗 Filtered interactions: {len(df_interactions_filtered):,}")
print(f"   (removed {len(df_interactions) - len(df_interactions_filtered):,} interactions)")

print("\n" + "="*60)


🔍 FILTERING TO USABLE DRUGS

✅ Drugs with SMILES: 12313
✅ Drugs in interaction network: 4567
✅ Usable drugs (BOTH SMILES + interactions): 3710

🔗 Filtered interactions: 2,323,786
   (removed 532,062 interactions)



In [3]:
# Cell 3: Create positive interaction set
print("="*60)
print("✅ CREATING POSITIVE INTERACTION SET")
print("="*60)

# Create set of positive pairs (both directions)
positive_pairs = set()

for _, row in df_interactions_filtered.iterrows():
    drug1, drug2 = row['drug1'], row['drug2']
    # Add both directions (undirected graph)
    positive_pairs.add((drug1, drug2))
    positive_pairs.add((drug2, drug1))

print(f"\n✅ Positive interaction pairs: {len(positive_pairs):,}")
print(f"   (includes both directions)")

# Create list of all usable drugs
usable_drugs_list = sorted(list(usable_drugs))
print(f"\n✅ Total usable drugs: {len(usable_drugs_list)}")

# Calculate possible pairs
possible_pairs = len(usable_drugs_list) * (len(usable_drugs_list) - 1)
print(f"✅ Total possible drug pairs: {possible_pairs:,}")

# Calculate sparsity
sparsity = (len(positive_pairs) / possible_pairs) * 100
print(f"✅ Graph sparsity: {sparsity:.4f}% (very sparse!)")

print("\n" + "="*60)


✅ CREATING POSITIVE INTERACTION SET

✅ Positive interaction pairs: 2,323,786
   (includes both directions)

✅ Total usable drugs: 3710
✅ Total possible drug pairs: 13,760,390
✅ Graph sparsity: 16.8875% (very sparse!)



In [4]:
# Cell 4: Generate negative samples (non-interacting pairs)
print("="*60)
print("🎲 GENERATING NEGATIVE SAMPLES")
print("="*60)

print("\n⏳ This will take ~1 minute...")

# We'll generate same number as positive samples for balance
num_negative_samples = len(positive_pairs)

negative_pairs = set()

print(f"\nTarget: {num_negative_samples:,} negative samples")

with tqdm(total=num_negative_samples, desc="Sampling negatives") as pbar:
    while len(negative_pairs) < num_negative_samples:
        # Randomly sample two different drugs
        drug1, drug2 = random.sample(usable_drugs_list, 2)
        
        # Check if this pair is NOT in positive pairs
        if (drug1, drug2) not in positive_pairs and (drug2, drug1) not in positive_pairs:
            negative_pairs.add((drug1, drug2))
            pbar.update(1)

print(f"\n✅ Generated {len(negative_pairs):,} negative samples")

# Verify no overlap
overlap = positive_pairs & negative_pairs
print(f"✅ Overlap check: {len(overlap)} (should be 0)")

print("\n📊 Dataset Balance:")
print(f"  • Positive pairs: {len(positive_pairs):,}")
print(f"  • Negative pairs: {len(negative_pairs):,}")
print(f"  • Ratio: 1:1 (balanced)")

print("\n" + "="*60)


🎲 GENERATING NEGATIVE SAMPLES

⏳ This will take ~1 minute...

Target: 2,323,786 negative samples


Sampling negatives: 2598206it [00:07, 336106.75it/s]                             


✅ Generated 2,323,786 negative samples
✅ Overlap check: 0 (should be 0)

📊 Dataset Balance:
  • Positive pairs: 2,323,786
  • Negative pairs: 2,323,786
  • Ratio: 1:1 (balanced)



In [6]:
# Cell 5: Create full labeled dataset (FAST VERSION)
print("="*60)
print("📋 CREATING FULL LABELED DATASET (OPTIMIZED)")
print("="*60)

# Create lookup dictionary for relation types (MUCH FASTER)
relation_lookup = {}
for _, row in df_interactions_filtered.iterrows():
    relation_lookup[(row['drug1'], row['drug2'])] = row['relation_type']
    relation_lookup[(row['drug2'], row['drug1'])] = row['relation_type']  # Both directions

print(f"✅ Created relation lookup with {len(relation_lookup):,} entries")

# Create positive samples with relation types
positive_data = []
for drug1, drug2 in tqdm(positive_pairs, desc="Processing positives"):
    relation = relation_lookup.get((drug1, drug2), 'unknown')
    positive_data.append({
        'drug1': drug1,
        'drug2': drug2,
        'label': 1,
        'relation_type': relation
    })

df_positive = pd.DataFrame(positive_data)
print(f"\n✅ Positive samples: {len(df_positive):,}")

# Create negative samples (no lookup needed)
negative_data = [
    {'drug1': d1, 'drug2': d2, 'label': 0, 'relation_type': 'none'}
    for d1, d2 in tqdm(negative_pairs, desc="Processing negatives")
]

df_negative = pd.DataFrame(negative_data)
print(f"✅ Negative samples: {len(df_negative):,}")

# Combine and shuffle
df_full = pd.concat([df_positive, df_negative], ignore_index=True)
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Total dataset: {len(df_full):,} samples")
print(f"\n📊 Label distribution:")
print(df_full['label'].value_counts())
print(f"\n📊 Relation type distribution:")
print(df_full['relation_type'].value_counts())

print("\n" + "="*60)


📋 CREATING FULL LABELED DATASET (OPTIMIZED)
✅ Created relation lookup with 2,323,786 entries


Processing positives: 100%|██████████| 2323786/2323786 [00:02<00:00, 997682.02it/s] 



✅ Positive samples: 2,323,786


Processing negatives: 100%|██████████| 2323786/2323786 [00:01<00:00, 2135461.99it/s]


✅ Negative samples: 2,323,786

✅ Total dataset: 4,647,572 samples

📊 Label distribution:
label
0    2323786
1    2323786
Name: count, dtype: int64

📊 Relation type distribution:
relation_type
none            2323786
synergistic     1412390
antagonistic     911396
Name: count, dtype: int64



In [7]:
# Cell 6: Create train/val/test splits
print("="*60)
print("✂️  CREATING TRAIN/VAL/TEST SPLITS")
print("="*60)

# Split: 70% train, 15% val, 15% test
train_size = int(0.70 * len(df_full))
val_size = int(0.15 * len(df_full))
test_size = len(df_full) - train_size - val_size

df_train = df_full[:train_size]
df_val = df_full[train_size:train_size+val_size]
df_test = df_full[train_size+val_size:]

print(f"\n📊 Split sizes:")
print(f"  • Train: {len(df_train):,} samples ({len(df_train)/len(df_full)*100:.1f}%)")
print(f"  • Val:   {len(df_val):,} samples ({len(df_val)/len(df_full)*100:.1f}%)")
print(f"  • Test:  {len(df_test):,} samples ({len(df_test)/len(df_full)*100:.1f}%)")

print(f"\n✅ Train set balance:")
print(df_train['label'].value_counts())

print(f"\n✅ Val set balance:")
print(df_val['label'].value_counts())

print(f"\n✅ Test set balance:")
print(df_test['label'].value_counts())

print("\n" + "="*60)


✂️  CREATING TRAIN/VAL/TEST SPLITS

📊 Split sizes:
  • Train: 3,253,300 samples (70.0%)
  • Val:   697,135 samples (15.0%)
  • Test:  697,137 samples (15.0%)

✅ Train set balance:
label
1    1626700
0    1626600
Name: count, dtype: int64

✅ Val set balance:
label
1    348934
0    348201
Name: count, dtype: int64

✅ Test set balance:
label
0    348985
1    348152
Name: count, dtype: int64



In [8]:
# Cell 7: Save train/val/test splits
print("="*60)
print("💾 SAVING TRAIN/VAL/TEST SPLITS")
print("="*60)

# Save splits
df_train.to_csv('../data/processed/train_split.csv', index=False)
df_val.to_csv('../data/processed/val_split.csv', index=False)
df_test.to_csv('../data/processed/test_split.csv', index=False)

print(f"\n✅ Saved train split: {len(df_train):,} samples")
print(f"✅ Saved val split: {len(df_val):,} samples")
print(f"✅ Saved test split: {len(df_test):,} samples")

# Also save the list of usable drugs
usable_drugs_df = df_drugs[df_drugs['drugbank_id'].isin(usable_drugs)]
usable_drugs_df.to_csv('../data/processed/usable_drugs.csv', index=False)
print(f"\n✅ Saved usable drugs info: {len(usable_drugs_df)} drugs")

print("\n💾 FILES CREATED:")
print("  ✅ train_split.csv")
print("  ✅ val_split.csv")
print("  ✅ test_split.csv")
print("  ✅ usable_drugs.csv")

print("\n" + "="*60)


💾 SAVING TRAIN/VAL/TEST SPLITS

✅ Saved train split: 3,253,300 samples
✅ Saved val split: 697,135 samples
✅ Saved test split: 697,137 samples

✅ Saved usable drugs info: 3710 drugs

💾 FILES CREATED:
  ✅ train_split.csv
  ✅ val_split.csv
  ✅ test_split.csv
  ✅ usable_drugs.csv



In [9]:
# Cell 8: STEP 2 FINAL SUMMARY
print("="*60)
print("✅ STEP 2: NEGATIVE SAMPLING & DATA SPLITS - COMPLETE!")
print("="*60)

print("\n📊 DATASET SUMMARY:")
print(f"  • Total samples: 4,647,572")
print(f"  • Positive (interactions): 2,323,786 (50%)")
print(f"  • Negative (non-interactions): 2,323,786 (50%)")

print("\n🔗 RELATION TYPES (in positives):")
print(f"  • Synergistic: 1,412,390 (60.8%)")
print(f"  • Antagonistic: 911,396 (39.2%)")

print("\n✂️  TRAIN/VAL/TEST SPLITS:")
print(f"  • Train: 3,253,300 (70%)")
print(f"  • Val:   697,135 (15%)")
print(f"  • Test:  697,137 (15%)")

print("\n💾 FILES SAVED:")
print(f"  ✅ train_split.csv")
print(f"  ✅ val_split.csv")
print(f"  ✅ test_split.csv")
print(f"  ✅ usable_drugs.csv")
print(f"  ✅ drugbank_interactions_typed.csv (from Step 1)")

print("\n✅ KEY IMPROVEMENTS FROM YOUR OLD NOTEBOOK:")
print(f"  • Proper negative sampling (was missing)")
print(f"  • Balanced dataset (was imbalanced)")
print(f"  • Validation set added (was only train/test)")
print(f"  • Multi-relational labels (synergistic/antagonistic)")

print("\n🎯 NEXT STEPS (STEP 3):")
print(f"  1. Generate enhanced molecular features (Morgan + descriptors)")
print(f"  2. Create node feature matrix")
print(f"  3. Build molecular graphs for each drug")
print(f"  4. Prepare data for PyTorch Geometric")

print("\n" + "="*60)



✅ STEP 2: NEGATIVE SAMPLING & DATA SPLITS - COMPLETE!

📊 DATASET SUMMARY:
  • Total samples: 4,647,572
  • Positive (interactions): 2,323,786 (50%)
  • Negative (non-interactions): 2,323,786 (50%)

🔗 RELATION TYPES (in positives):
  • Synergistic: 1,412,390 (60.8%)
  • Antagonistic: 911,396 (39.2%)

✂️  TRAIN/VAL/TEST SPLITS:
  • Train: 3,253,300 (70%)
  • Val:   697,135 (15%)
  • Test:  697,137 (15%)

💾 FILES SAVED:
  ✅ train_split.csv
  ✅ val_split.csv
  ✅ test_split.csv
  ✅ usable_drugs.csv
  ✅ drugbank_interactions_typed.csv (from Step 1)

✅ KEY IMPROVEMENTS FROM YOUR OLD NOTEBOOK:
  • Proper negative sampling (was missing)
  • Balanced dataset (was imbalanced)
  • Validation set added (was only train/test)
  • Multi-relational labels (synergistic/antagonistic)

🎯 NEXT STEPS (STEP 3):
  1. Generate enhanced molecular features (Morgan + descriptors)
  2. Create node feature matrix
  3. Build molecular graphs for each drug
  4. Prepare data for PyTorch Geometric

